In [1]:
import json
import folium
import geonamescache
from IPython.display import display

In [2]:
def find_city(query: str) -> dict | None:
    """
    Search the GeoNames database for a city by name (case-insensitive).
    Also checks alternate/translated names for better coverage.
    When multiple cities share a name, the most populous one is returned.
    """
    gc = geonamescache.GeonamesCache()
    all_cities = gc.get_cities().values()
    query_lower = query.strip().lower()
    candidates = []

    for city in all_cities:
        if city["name"].lower() == query_lower:
            candidates.append(city)
            continue
        if any(alt.lower() == query_lower for alt in city.get("alternatenames", [])):
            candidates.append(city)

    if not candidates:
        return None
    return max(candidates, key=lambda c: c.get("population", 0))


def city_dict(query: str) -> dict:
    """Return {"name", "lat", "lng"} for a given city name."""
    city = find_city(query)
    if city is None:
        raise ValueError(f"City not found: '{query}'")
    return {
        "name": city["name"],
        "lat":  city["latitude"],
        "lng":  city["longitude"],
    }


def preview_map(result: dict) -> folium.Map:
    """Render an interactive map centred on the city with a labelled marker."""
    lat, lng, name = result["lat"], result["lng"], result["name"]
    m = folium.Map(location=[lat, lng], zoom_start=10, control_scale=True)
    folium.Marker(
        location=[lat, lng],
        popup=folium.Popup(
            f"<b>{name}</b><br>lat: {lat}<br>lng: {lng}",
            max_width=200
        ),
        tooltip=name,
        icon=folium.Icon(color="red", icon="map-marker"),
    ).add_to(m)
    return m

## 🔍 Look up a city

Edit `city_name` below and run the cell.

In [6]:
city_name = "Lisbon"   # <- change this

result = city_dict(city_name)
print(json.dumps(result, indent=4, ensure_ascii=False))
preview_map(result)

{
    "name": "Lisbon",
    "lat": 38.72509,
    "lng": -9.1498
}


## 📋 Look up multiple cities at once

In [4]:
cities_to_lookup = ["Tokyo", "New York", "Bologna", "Sao Paulo", "Paris"]

results = []
for name in cities_to_lookup:
    try:
        results.append(city_dict(name))
    except ValueError as e:
        print(f"Warning: {e}")

print(json.dumps(results, indent=4, ensure_ascii=False))

# Multi-city map centred on the first result
multi_map = folium.Map(location=[results[0]["lat"], results[0]["lng"]], zoom_start=2)
for r in results:
    folium.Marker(
        location=[r["lat"], r["lng"]],
        popup=folium.Popup(
            "<b>" + r['name'] + "</b><br>lat: " + str(r['lat']) + "<br>lng: " + str(r['lng']),
            max_width=200
        ),
        tooltip=r["name"],
        icon=folium.Icon(color="blue", icon="map-marker"),
    ).add_to(multi_map)
multi_map

[
    {
        "name": "Tokyo",
        "lat": 35.6895,
        "lng": 139.69171
    },
    {
        "name": "New York City",
        "lat": 40.71427,
        "lng": -74.00597
    },
    {
        "name": "Bologna",
        "lat": 44.49381,
        "lng": 11.33875
    },
    {
        "name": "São Paulo",
        "lat": -23.5475,
        "lng": -46.63611
    },
    {
        "name": "Paris",
        "lat": 48.85341,
        "lng": 2.3488
    }
]
